# Day 04 — контролируемые эксперименты

## D4-01 — гипотезы, протокол и проверка reference

**Цель.** Загрузить настоящие Day02 inputs и Day03 reference, проверить их происхождение и увидеть baseline, с которым будут сравниваться эксперименты. Выход — проверенный контекст `(N, columns)` и сохранённые validation metrics. В этой ячейке нет обучения.

**Краткая теория.** Проверим три независимые гипотезы: word unigrams; замена LogisticRegression на LinearSVC с тем же TF-IDF; подбор только `LogisticRegression.C ∈ {0.1, 1, 10}`. Каждый вариант сравнивается с baseline на одних трёх `StratifiedGroupKFold` folds только внутри Day03 train. После выбора по среднему fold macro F1 выбранный pipeline будет fit на полном train и один раз проверен на прежней validation. Test не участвует в поиске и остаётся для Day05.

Pipeline заново обучает TF-IDF внутри train части каждого fold, чтобы validation части fold не влияли на vocabulary/IDF. Метрики выбора используют три класса (`labels=[0, 1, 2]`); fold standard deviation описывает разброс, но не является оценкой статистической значимости.

In [1]:
from hashlib import sha256
import json
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display


def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "SPEC.md").is_file() and (candidate / "src/finnews_sentiment").is_dir():
            return candidate
    raise FileNotFoundError("Project root with SPEC.md and src/finnews_sentiment not found")


ROOT: Path = find_project_root(Path.cwd().resolve())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from finnews_sentiment.data.baseline_config import load_baseline_config
from finnews_sentiment.data.load_data import CLASS_ORDER
from finnews_sentiment.data.load_processed import load_processed_news

print(f"Project root: {ROOT}")
print(f"Class order: {CLASS_ORDER}")

Project root: /home/makarlistkov/projects/Text-classification
Class order: ('negative', 'neutral', 'positive')


### Проверка данных и Day03 artifacts

Загрузчик проверяет схему, label mapping и Day02 provenance/checksum. Далее сверим контрольные суммы Day03, manifest и порядок validation predictions. Если артефактов нет, нужно отдельно воспроизвести Day02 и Day03 по инструкциям в `docs/day02.md` и `docs/day03.md`; notebook не скачивает данные и не подменяет их fixtures.

In [2]:
config_path = ROOT / "configs/train_baseline.json"
config = load_baseline_config(config_path)
processed_news = load_processed_news(
    ROOT / config.input_path,
    ROOT / config.provenance_path,
    ROOT / "configs/dataset.json",
)

report_dir = ROOT / "reports/day03"
manifest_path = ROOT / config.split_output_dir / "splits.csv"
artifact_paths = {
    "manifest": manifest_path,
    "split_summary": report_dir / "split_summary.json",
    "predictions": report_dir / "validation_predictions.csv",
    "metrics": report_dir / "metrics.json",
    "effective_config": report_dir / "config.json",
}
provenance = json.loads((report_dir / "provenance.json").read_text(encoding="utf-8"))
for artifact_name, artifact_path in artifact_paths.items():
    expected_hash = provenance["checksums"][artifact_name]
    actual_hash = sha256(artifact_path.read_bytes()).hexdigest()
    if actual_hash != expected_hash:
        raise ValueError(f"Day03 {artifact_name} checksum mismatch")

manifest = pd.read_csv(manifest_path, dtype={"source_row_id": "int64", "group_id": "int64"})
predictions = pd.read_csv(
    artifact_paths["predictions"],
    dtype={"source_row_id": "int64", "true_label_id": "int64", "baseline_pred_label_id": "int64", "dummy_pred_label_id": "int64"},
)
split_summary = json.loads(artifact_paths["split_summary"].read_text(encoding="utf-8"))
metrics = json.loads(artifact_paths["metrics"].read_text(encoding="utf-8"))
effective_config = json.loads(artifact_paths["effective_config"].read_text(encoding="utf-8"))

if tuple(manifest.columns) != ("source_row_id", "group_id", "split"):
    raise ValueError("Day03 split manifest schema mismatch")
if tuple(predictions.columns) != ("source_row_id", "true_label_id", "baseline_pred_label_id", "dummy_pred_label_id"):
    raise ValueError("Day03 validation predictions schema mismatch")
expected_valid_ids = manifest.loc[manifest["split"] == "validation", "source_row_id"].tolist()
if predictions["source_row_id"].tolist() != expected_valid_ids:
    raise ValueError("Day03 validation predictions do not align with manifest")
if len(processed_news) != len(manifest) or manifest["source_row_id"].tolist() != processed_news["source_row_id"].tolist():
    raise ValueError("Day03 manifest does not cover processed rows in canonical order")
if metrics["evaluation_split"] != "validation" or metrics["class_order"] != list(CLASS_ORDER):
    raise ValueError("Day03 metrics split or class order mismatch")
if metrics["n_valid"] != len(predictions):
    raise ValueError("Day03 metric denominator does not match predictions")
if effective_config["random_state"] != 42 or effective_config["split_algorithm"] != "group-stratified-v1":
    raise ValueError("Day03 effective split settings differ from the agreed reference")

split_sizes = manifest["split"].value_counts().reindex(["train", "validation", "test"], fill_value=0)
print(f"Processed input: {processed_news.shape}; manifest: {manifest.shape}")
print(f"Group-aware split rows: {split_sizes.to_dict()}")
print(f"Day03 sklearn version: {provenance['versions']['scikit-learn']}")
display(pd.DataFrame([
    {"model": name, "accuracy": metrics[name]["accuracy"], "macro_f1": metrics[name]["macro_f1"], "n_valid": metrics[name]["n_samples"]}
    for name in ("baseline", "dummy")
]).set_index("model"))

Processed input: (3453, 5); manifest: (3453, 3)
Group-aware split rows: {'train': 2210, 'validation': 553, 'test': 690}
Day03 sklearn version: 1.9.1


,accuracy,macro_f1,n_valid
model,,,
baseline,0.813743,0.720965,553
dummy,0.622061,0.255667,553


### Результат и интерпретация

В сохранённой Day03 reference baseline обучался только на train, а его validation macro F1 был `0.720965` (accuracy `0.813743`). Dummy most-frequent дал macro F1 `0.255667` (accuracy `0.622061`). На validation больше neutral примеров, поэтому accuracy и macro F1 отвечают на разные вопросы; далее решения сравниваем по заранее выбранной macro F1.

Baseline — допустимый победитель Day04. CV добавляет оценку на нескольких group-aware разбиениях и выбирает настройки без использования validation; validation затем служит единственным подтверждением выбранного варианта на прежнем split. Ни CV, ни одна validation оценка не обещают улучшения на новых данных.

### Самопроверка

1. Что меняется в каждой из трёх гипотез, а что остаётся фиксированным?
2. Почему vocabulary и IDF нужно fit-ить отдельно внутри train части каждого fold?
3. Почему validation может подтвердить выбор, но не должна переизбирать конфигурацию?
4. Почему baseline может остаться лучшим, даже если проверены три направления?